# SMS Spam Classifier — Kaggle Notebook
Dataset: `uciml/sms-spam-collection-dataset`

This trains TF-IDF + Logistic Regression and saves compact JSON weights.

In [ ]:
import re, time, json, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

CLEAN_URL = re.compile(r"https?://\S+")
CLEAN_NUM = re.compile(r"\b\d+\b")
def clean_text(s):
    s = (s or "").lower()
    s = CLEAN_URL.sub(" url ", s)
    s = CLEAN_NUM.sub(" num ", s)
    return s.strip()

import os
import pandas as pd
try:
    df = pd.read_csv('/kaggle/input/sms-spam-collection-dataset/spam.csv', encoding='latin1')
    df = df.rename(columns={'v1':'label','v2':'text'})[['label','text']].dropna()
    df['label'] = df['label'].map(lambda x: 'spam' if str(x).lower().strip()=='spam' else 'ham')
except Exception as e:
    raise
print(df.shape, df.head())

X = df['text'].map(clean_text).tolist()
y = (df['label']=='spam').astype(int).values

# Baseline
vect1 = TfidfVectorizer(ngram_range=(1,1), min_df=2, max_df=0.98, sublinear_tf=True, norm='l2')
X1 = vect1.fit_transform(X)
Xtr,Xte,ytr,yte = train_test_split(X1,y,test_size=0.2,random_state=42,stratify=y)
clf1 = LogisticRegression(max_iter=200, C=1.0, solver='liblinear')
clf1.fit(Xtr,ytr)
pred1 = clf1.predict(Xte)
print('Baseline F1:', f1_score(yte,pred1))

# Improved
vect = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.98, sublinear_tf=True, norm='l2')
Xv = vect.fit_transform(X)
Xtr,Xte,ytr,yte = train_test_split(Xv,y,test_size=0.2,random_state=42,stratify=y)
clf = LogisticRegression(max_iter=200, C=2.0, solver='liblinear')
clf.fit(Xtr,ytr)
pred = clf.predict(Xte)
print('Improved F1:', f1_score(yte,pred))
print(classification_report(yte,pred,digits=4))


model_json = {
  'model_type':'logreg','version':'v1.0.0','classes':['ham','spam'],
  'coef': clf.coef_.reshape(-1).tolist(),
  'intercept': float(clf.intercept_[0]),
  'vocabulary': {tok:int(idx) for tok,idx in vect.vocabulary_.items()},
  'idf': vect.idf_.tolist(),
  'tfidf': {'ngram_range': vect.ngram_range, 'sublinear_tf': True, 'norm':'l2', 'use_idf': True, 'min_df': vect.min_df, 'max_df': float(vect.max_df), 'lowercase': True}
}
with open('/kaggle/working/model.pkl','w') as f:
    json.dump(model_json,f)
with open('/kaggle/working/feature_config.json','w') as f:
    json.dump({'token_pattern': r"[a-zA-Z']+", 'lowercase': True, 'strip': True}, f, indent=2)
with open('/kaggle/working/label_map.json','w') as f:
    json.dump({'0':'ham','1':'spam'}, f, indent=2)
print('Saved /kaggle/working/*.json')
